In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.model_selection import GroupKFold, cross_val_score, train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb

In [2]:
df = pd.read_csv('Merged_SoilMoisture_Data.csv')

In [3]:
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Points: {df['Point_ID'].nunique()}  |  Dates: {df['Date'].nunique()}")

Loaded: 401 rows × 17 columns
Points: 100  |  Dates: 22


In [4]:
def engineer_features(df):
    df = df.copy()
 
    #  Temporal features (captures seasonal SM cycle) 
    df['month']     = df['Date'].dt.month
    df['doy']       = df['Date'].dt.dayofyear
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
 
    #  Fix 3 missing SAR values (spatial median per point) 
    for col in ['VV_500m', 'VH_500m', 'Angle_500m']:
        null_mask = df[col].isnull()
        if null_mask.sum() > 0:
            fill = df.groupby('Point_ID')[col].transform('median')
            df.loc[null_mask, col] = fill[null_mask]
 
    #  SAR physics features 
    df['VV_linear']  = 10 ** (df['VV_500m'] / 10)   # dB → linear
    df['VH_linear']  = 10 ** (df['VH_500m'] / 10)
    df['VV_VH_diff'] = df['VV_500m'] - df['VH_500m']
 
    #  Terrain interaction features 
    df['slope_elev']  = df['Elevation_m'] / (df['Slope'] + 0.1)
    df['rough_slope'] = df['Roughness'] * df['Slope']
 
    df['VV_x_season'] = df['VV_linear'] * df['month_sin']
    df['VH_x_season'] = df['VH_linear'] * df['month_sin']
    df['VV_x_cos']    = df['VV_linear'] * df['month_cos']
    df['VH_x_cos']    = df['VH_linear'] * df['month_cos']
 
    return df
 
df = engineer_features(df)
 
FEATURES = [
    # SAR backscatter (raw + linear)
    'VV_500m', 'VH_500m', 'Angle_500m',
    'VV_linear', 'VH_linear', 'VV_VH_diff',
    # Polarimetric indices
    'CR=VH/VV', 'RVI', 'RPI=VV/VH', 'NDPI=(VV-VH)/(VV+VH)', 'MPI',
    # Terrain
    'Elevation_m', 'Slope', 'Aspect', 'Roughness',
    'slope_elev', 'rough_slope',
    # Temporal
    'doy','VV_x_season',
    'VH_x_season',
    'VV_x_cos',   
    'VH_x_cos'    
 
    
]
 
X = df[FEATURES].copy()
y = df['Observed_SM'].copy()

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

X = df[FEATURES].copy()
y = df['Observed_SM'].copy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42   # 0.2 instead of 0.25 — more training data
)

# Scale
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Tighter grid tuned for small datasets
param_grid = {
    'n_estimators':     [50,100,200,300,500],
    'max_depth':        [2,4,8],
    'min_samples_leaf': [4,6,8,10, 12],   # higher — prevents overfitting on small data
    'max_features':     ['sqrt', 0.5, 0.7,1],
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator  = rf,
    param_grid = param_grid,
    cv         = 5,           # 5-fold instead of 10 — more data per fold
    scoring    = 'r2',
    verbose    = 1,
    n_jobs     = -1
)

grid_search.fit(X_train_sc, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV R2     :", round(grid_search.best_score_, 4))

best_rf = grid_search.best_estimator_

# CV
cv_scores = cross_val_score(best_rf, X_train_sc, y_train, cv=5, scoring='r2')
print(f"\nCV R2 Scores : {cv_scores.round(4)}")
print(f"CV Mean R2   : {cv_scores.mean():.4f}")
print(f"CV Std R2    : {cv_scores.std():.4f}")

# Predictions
y_pred_train = best_rf.predict(X_train_sc)
y_pred_test  = best_rf.predict(X_test_sc)

# Evaluation
def evaluate(y_true, y_pred, split_name):
    r      = np.corrcoef(y_true, y_pred)[0, 1]
    r2     = r2_score(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    bias   = np.mean(y_pred - y_true)
    ubrmse = np.sqrt(np.mean(((y_pred - y_pred.mean()) - (y_true - y_true.mean())) ** 2))

    print(f"\n── {split_name} ──")
    print(f"  R      : {r:.4f}")
    print(f"  R2     : {r2:.4f}")
    print(f"  RMSE   : {rmse:.4f}")
    print(f"  ubRMSE : {ubrmse:.4f}")
    print(f"  MAE    : {mae:.4f}")
    print(f"  Bias   : {bias:.4f}")

evaluate(y_train, y_pred_train, "Training")
evaluate(y_test,  y_pred_test,  "Testing")

Fitting 5 folds for each of 300 candidates, totalling 1500 fits
Best Parameters: {'max_depth': 8, 'max_features': 0.7, 'min_samples_leaf': 6, 'n_estimators': 200}
Best CV R2     : 0.3456

CV R2 Scores : [0.2921 0.4617 0.3102 0.4191 0.2449]
CV Mean R2   : 0.3456
CV Std R2    : 0.0814

── Training ──
  R      : 0.8502
  R2     : 0.6758
  RMSE   : 0.0466
  ubRMSE : 0.0466
  MAE    : 0.0366
  Bias   : -0.0001

── Testing ──
  R      : 0.6069
  R2     : 0.3633
  RMSE   : 0.0646
  ubRMSE : 0.0644
  MAE    : 0.0518
  Bias   : 0.0056


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

X = df[FEATURES].copy()
y = df['Observed_SM'].copy()
groups = df['Point_ID'].values   # use spatial grouping

gkf = GroupKFold(n_splits=5)

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    bootstrap=True
)

param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6],
    'min_samples_leaf': [4, 6, 8, 10, 12],
    'min_samples_split': [8, 10, 12, 15],
    'max_features': [0.3, 0.4, 0.5, 'sqrt'],
    'max_samples': [0.6, 0.7, 0.8, 0.9]
}

search = RandomizedSearchCV(
    rf,
    param_distributions=param_grid,
    n_iter=30,
    scoring='r2',
    cv=gkf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X, y, groups=groups)

best_rf = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV R2 :", round(search.best_score_, 4))

cv_scores = cross_val_score(
    best_rf, X, y,
    cv=gkf,
    groups=groups,
    scoring='r2',
    n_jobs=-1
)

print("\nCV R2 Scores:", np.round(cv_scores, 4))
print("CV Mean R2  :", round(cv_scores.mean(), 4))
print("CV Std R2   :", round(cv_scores.std(), 4))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_samples': 0.9, 'max_features': 0.5, 'max_depth': 5}
Best CV R2 : 0.341

CV R2 Scores: [0.4475 0.5265 0.1355 0.3025 0.2929]
CV Mean R2  : 0.341
CV Std R2   : 0.1355
